# Combined YieldSAT + Morocco Crop Classification Baseline

This notebook loads both the YieldSAT feature store (notebook 05/06) and the Morocco enriched dataset (notebook 07/08), aligns them to a five-label schema (`Wheat`, `Corn`, `Alfalfa`, `Other crop`, `Background`), extracts an AOI-compatible spectral feature core shared by both datasets, and trains XGBoost, LightGBM, and CatBoost on the full combined data.

**Five-label alignment:**
- YieldSAT `wheat` -> `Wheat`, `corn` -> `Corn`
- YieldSAT `soybean` and `rapeseed` are removed from `Other crop` because subtype diagnostics showed AOI seasons were closer to them than to known target centroids
- Morocco `Beets`, `Parsley`, and `Peas` are removed from `Other crop` for this conservative Corn-cleaning experiment because their subtype centroids/neighbor overlap were Corn-like
- Remaining Morocco `Other crop` stays in the taxonomy and is downweighted during model fitting
- Morocco `Alfalfa` remains Morocco-only
- Morocco `Background` keeps all original background subclasses, including `Bare soil`, `Water`, and `Weed`

**Hybrid Conservative spectral experiment:** train on 9 vegetation indices plus 4 red-edge/NIR brightness-normalized bands (`B06`, `B07`, `B08`, `B8A`). Visible normalized bands (`B02`, `B03`, `B04`, `B05`) and SWIR (`B11`) are excluded because prior AOI comparisons showed they remained shifted and pushed predictions toward Background/Wheat.

**Other-crop cleaning experiment:** remove `soybean`, `rapeseed`, `Beets`, `Parsley`, and `Peas`, then downweight the remaining `Other crop` class by `OTHER_CROP_WEIGHT_MULTIPLIER = 0.5`. This keeps remaining Morocco `Other crop` as hard negatives while reducing target-like other-crop dominance.

**Common feature core:** vegetation index means derivable from default AOI bands (9) plus selected brightness-normalized default AOI bands (4). This spectral experiment intentionally excludes all weather features. The model also excludes `area_ha`, `doy`, `observation_count`, and S2 features that require bands not downloaded by default (`B01`, `B09`, `B12`, plus dependent CAI and NBR).

**Data prerequisites:**
- YieldSAT feature store: `crop_mapping_fields.parquet` + `crop_mapping_observations.parquet` (from notebook 05)
- Morocco features: `features.parquet` (from notebooks 07 -> 08)

This code runs in Colab, Kaggle, or a local environment with the data paths configured below.


## 1. Install Dependencies

In [1]:
# %pip install -q pyarrow scikit-learn xgboost lightgbm catboost
# %pip install -q pandas numpy matplotlib

## 2. Imports and Configuration

In [2]:
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [3]:
# --- Configuration ---
# Update these paths for your environment.
YIELDSAT_DIR = Path("data/yieldsat_crop_mapping_feature_store")
MOROCCO_PATH = Path("data/morocco_crop_irrigation/features.parquet")

# Colab fallback
if not YIELDSAT_DIR.exists():
    YIELDSAT_DIR = Path("/content/yieldsat_crop_mapping_feature_store")
if not MOROCCO_PATH.exists():
    MOROCCO_PATH = Path("/content/data/morocco_crop_irrigation/features.parquet")

FIELDS_PATH = YIELDSAT_DIR / "crop_mapping_fields.parquet"
OBSERVATIONS_PATH = YIELDSAT_DIR / "crop_mapping_observations.parquet"

print("YieldSAT fields:", FIELDS_PATH)
print("YieldSAT observations:", OBSERVATIONS_PATH)
print("Morocco features:", MOROCCO_PATH)
print("Files exist:", FIELDS_PATH.exists(), OBSERVATIONS_PATH.exists(), MOROCCO_PATH.exists())

YieldSAT fields: data\yieldsat_crop_mapping_feature_store\crop_mapping_fields.parquet
YieldSAT observations: data\yieldsat_crop_mapping_feature_store\crop_mapping_observations.parquet
Morocco features: data\morocco_crop_irrigation\features.parquet
Files exist: True True True


## 3. Load and Prepare YieldSAT

Loads the two-file feature store, selects wheat/corn/other+ labels, and aggregates observations to one row per field using only the mean (to match the Morocco feature set).

In [4]:
fields = pd.read_parquet(FIELDS_PATH)
observations = pd.read_parquet(OBSERVATIONS_PATH)

fields["crop"] = fields["crop"].astype(str).str.lower()
fields["country"] = fields.get("country", fields.get("adm_country", "unknown")).fillna("unknown").astype(str)
fields["year"] = fields["year"].astype(str)

print("Fields:", fields.shape)
print("Observations:", observations.shape)
print("Crop counts:")
print(fields["crop"].value_counts())

Fields: (2173, 935)
Observations: (100670, 258)
Crop counts:
crop
soybean     1305
wheat        454
corn         303
rapeseed     111
Name: count, dtype: int64


In [5]:
S2_REFLECTANCE_COLS = [
    col for col in observations.columns
    if col.startswith("s2_B")
    and any(col.endswith(suffix) for suffix in ("_mean", "_std", "_min", "_max", "_p10", "_p25", "_p75", "_p90"))
]

observations[S2_REFLECTANCE_COLS] = observations[S2_REFLECTANCE_COLS].mask(
    observations[S2_REFLECTANCE_COLS] > 2.0
)

EVI_STAT_COLS = [
    col for col in observations.columns
    if col.startswith("s2_EVI_")
    and not col.endswith("_count")
]
for col in EVI_STAT_COLS:
    observations[col] = observations[col].clip(-1.0, 1.0)

print(f"Cleaned {len(S2_REFLECTANCE_COLS)} reflectance columns (>2.0 → NaN)")
print(f"Clipped {len(EVI_STAT_COLS)} EVI columns to [-1, 1]")

Cleaned 104 reflectance columns (>2.0 → NaN)
Clipped 8 EVI columns to [-1, 1]


In [6]:
OTHER_SOURCE_CROPS = ["soybean", "rapeseed"]

wheat_df = fields[fields["crop"] == "wheat"].copy()
corn_df = fields[fields["crop"] == "corn"].copy()
other_candidates = fields[fields["crop"].isin(OTHER_SOURCE_CROPS)].copy()

n_other = len(wheat_df)
print(f"Wheat: {len(wheat_df)}, Corn: {len(corn_df)}, Other candidates: {len(other_candidates)}")
print(f"Sampling {n_other} other-class fields balanced by country")

countries = sorted(other_candidates["country"].unique())
per_country = n_other // len(countries)
remainder = n_other % len(countries)
sampled_parts = []
for idx, country in enumerate(countries):
    cc = other_candidates[other_candidates["country"] == country]
    take = per_country + (1 if idx < remainder else 0)
    take = min(take, len(cc))
    if take > 0:
        sampled_parts.append(cc.sample(n=take, random_state=RANDOM_SEED + idx))


def add_columns_once(df: pd.DataFrame, **columns) -> pd.DataFrame:
    extra = pd.DataFrame(columns, index=df.index)
    return pd.concat([df.copy(), extra], axis=1)


other_df = pd.concat(sampled_parts, ignore_index=True).copy()
wheat_df = add_columns_once(wheat_df, label="Wheat", source_crop=wheat_df["crop"].to_numpy())
corn_df = add_columns_once(corn_df, label="Corn", source_crop=corn_df["crop"].to_numpy())
other_df = add_columns_once(other_df, label="Other crop", source_crop=other_df["crop"].to_numpy())

yieldsat_fields = pd.concat([wheat_df, corn_df, other_df], ignore_index=True)
yieldsat_fields = yieldsat_fields.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

print("Selected YieldSAT fields:", yieldsat_fields.shape)
print(yieldsat_fields["label"].value_counts())
print("Label x country:")
print(pd.crosstab(yieldsat_fields["label"], yieldsat_fields["country"]))
print("Other crop source crops:")
print(yieldsat_fields.loc[yieldsat_fields["label"] == "Other crop", "source_crop"].value_counts())



Wheat: 454, Corn: 303, Other candidates: 1416
Sampling 454 other-class fields balanced by country
Selected YieldSAT fields: (1209, 937)
label
Wheat         454
Other crop    452
Corn          303
Name: count, dtype: int64
Label x country:
country     Argentina  Brazil  Germany  Uruguay
label                                          
Corn              185     118        0        0
Other crop        114     114      111      113
Wheat             126     140      188        0
Other crop source crops:
source_crop
soybean     341
rapeseed    111
Name: count, dtype: int64


In [7]:
def is_leakage_column(col: str) -> bool:
    lower = col.lower()
    return any(term in lower for term in ("yield", "target")) or lower == "crop"


selected_ids = set(yieldsat_fields["field_id"].astype(str))
obs = observations[observations["field_id"].astype(str).isin(selected_ids)].copy()
obs["field_id"] = obs["field_id"].astype(str)
yieldsat_fields["field_id"] = yieldsat_fields["field_id"].astype(str)

for col in obs.columns:
    if pd.api.types.is_bool_dtype(obs[col]):
        obs[col] = obs[col].astype("int8")

obs_excluded = {"field_id", "date", "crop", "label"}
obs_feature_cols = [
    col for col in obs.columns
    if col not in obs_excluded and not is_leakage_column(col)
    and (pd.api.types.is_bool_dtype(obs[col]) or pd.api.types.is_numeric_dtype(obs[col]))
]

obs_count = obs.groupby("field_id").size().rename("observation_count")
obs_mean = obs.groupby("field_id")[obs_feature_cols].mean().add_prefix("obs_mean__")

obs_agg = pd.concat([obs_count, obs_mean], axis=1).reset_index()

yieldsat_model = yieldsat_fields[["field_id", "label", "source_crop", "country", "year"]].merge(obs_agg, on="field_id", how="inner")
yieldsat_model = yieldsat_model.replace([np.inf, -np.inf], np.nan)

print("YieldSAT model table:", yieldsat_model.shape)
print("Rows lost:", len(yieldsat_fields) - len(yieldsat_model))
print("Label counts:")
print(yieldsat_model["label"].value_counts())
print("Other crop source crops:")
print(yieldsat_model.loc[yieldsat_model["label"] == "Other crop", "source_crop"].value_counts())


YieldSAT model table: (1209, 262)
Rows lost: 0
Label counts:
label
Wheat         454
Other crop    452
Corn          303
Name: count, dtype: int64
Other crop source crops:
source_crop
soybean     341
rapeseed    111
Name: count, dtype: int64


## 4. Load and Prepare Morocco

Loads the enriched `features.parquet`, collapses duplicate parcel-date observations, clips unstable EVI, and aggregates to one row per parcel (mean only for common-core features).

In [8]:
morocco_ts = pd.read_parquet(MOROCCO_PATH)

print("Morocco loaded:", morocco_ts.shape)
print("Unique parcels:", morocco_ts["parcel_id"].nunique())
print("Original label counts:")
print(morocco_ts.drop_duplicates("parcel_id")["label"].value_counts())
print("Model label counts:")
print(morocco_ts.drop_duplicates("parcel_id")["model_label"].value_counts())



Morocco loaded: (390766, 70)
Unique parcels: 7734
Original label counts:
label
Wheat         2614
Corn          1252
Bare soil      942
Alfalfa        857
Beets          424
Potatoes       416
Weed           346
Peas           229
Peanut         167
Onions         149
Tomatoes       128
Sugarcane       94
Fava Beans      44
Parsley         31
Rice            25
Water           16
Name: count, dtype: int64
Model label counts:
model_label
Wheat         2614
Other crop    1707
Background    1304
Corn          1252
Alfalfa        857
Name: count, dtype: int64


In [9]:
morocco_ts["satellite_date"] = pd.to_datetime(morocco_ts["satellite_date"], errors="coerce")

rows_before = len(morocco_ts)

if "s2_EVI_mean" in morocco_ts.columns:
    morocco_ts["s2_EVI_mean"] = morocco_ts["s2_EVI_mean"].clip(-1.0, 1.0)

numeric_columns = morocco_ts.select_dtypes(include=[np.number]).columns.tolist()
metadata_columns = [col for col in morocco_ts.columns if col not in numeric_columns]

agg_spec = {col: "mean" for col in numeric_columns}
agg_spec.update({col: "first" for col in metadata_columns if col not in {"parcel_id", "satellite_date", "field_id"}})

morocco_ts = (
    morocco_ts
    .sort_values(["parcel_id", "satellite_date"])
    .groupby(["parcel_id", "satellite_date"], as_index=False)
    .agg(agg_spec)
)

print("Rows before cleanup:", rows_before)
print("Rows after duplicate-date removal:", len(morocco_ts))
print("Unique parcels:", morocco_ts["parcel_id"].nunique())

Rows before cleanup: 390766
Rows after duplicate-date removal: 309914
Unique parcels: 7734


In [10]:
excluded_from_agg = {
    "year", "centroid_lon", "centroid_lat",
    "bbox_min_lon", "bbox_min_lat", "bbox_max_lon", "bbox_max_lat",
    "area_ha",
}

base_feature_cols = [
    col for col in morocco_ts.select_dtypes(include=[np.number]).columns
    if col not in excluded_from_agg
]

morocco_agg = morocco_ts.groupby("parcel_id")[base_feature_cols].mean().reset_index()

coverage = (
    morocco_ts.groupby("parcel_id")
    .agg(
        observation_count=("satellite_date", "count"),
        first_obs=("satellite_date", "min"),
        last_obs=("satellite_date", "max"),
    )
    .reset_index()
)
coverage["time_span_days"] = (coverage["last_obs"] - coverage["first_obs"]).dt.days
coverage = coverage[["parcel_id", "observation_count", "time_span_days"]]

metadata_cols = ["parcel_id", "label", "model_label", "irrigation_type", "survey_zone", "area_ha"]
morocco_meta = (
    morocco_ts[metadata_cols]
    .drop_duplicates("parcel_id")
    .rename(columns={"label": "source_crop", "model_label": "label"})
)

morocco_model = morocco_meta.merge(morocco_agg, on="parcel_id", how="inner").merge(coverage, on="parcel_id", how="inner")

print("Morocco model table:", morocco_model.shape)
print("Label counts:")
print(morocco_model["label"].value_counts())
print("Other crop source crops:")
print(morocco_model.loc[morocco_model["label"] == "Other crop", "source_crop"].value_counts())



Morocco model table: (7734, 59)
Label counts:
label
Wheat         2614
Other crop    1707
Background    1304
Corn          1252
Alfalfa        857
Name: count, dtype: int64
Other crop source crops:
source_crop
Beets         424
Potatoes      416
Peas          229
Peanut        167
Onions        149
Tomatoes      128
Sugarcane      94
Fava Beans     44
Parsley        31
Rice           25
Name: count, dtype: int64


## 5. Subsample Morocco by Zone

Keeps all Alfalfa parcels. For all other Morocco labels, takes a random 60% per survey_zone to reduce Morocco dominance while preserving geographic distribution.

In [11]:
import math

alfalfa = morocco_model[morocco_model["label"] == "Alfalfa"]
others = morocco_model[morocco_model["label"] != "Alfalfa"]

print(f"Alfalfa kept: {len(alfalfa)}")
print(f"Others before sampling: {len(others)}")

sampled_parts = []
for zone, group in sorted(others.groupby("survey_zone")):
    n = int(math.ceil(len(group) * 0.6))
    part = group.sample(n=n, random_state=RANDOM_SEED)
    sampled_parts.append(part)
    print(f"  {zone}: {len(group)} -> {n} ({n/len(group)*100:.0f}%)")

others_sampled = pd.concat(sampled_parts, ignore_index=True)
morocco_model = pd.concat([alfalfa, others_sampled], ignore_index=True)

print(f"\nMorocco after per-zone 60% sampling: {len(morocco_model)}")
print("Label counts:")
print(morocco_model["label"].value_counts())

Alfalfa kept: 857
Others before sampling: 6877
  Doukkala: 3377 -> 2027 (60%)
  Gharb: 1229 -> 738 (60%)
  Souss: 163 -> 98 (60%)
  Statte: 231 -> 139 (60%)
  Tadla: 1045 -> 627 (60%)
  el haouz: 832 -> 500 (60%)

Morocco after per-zone 60% sampling: 4986
Label counts:
label
Wheat         1564
Other crop     989
Alfalfa        857
Background     812
Corn           764
Name: count, dtype: int64


## 6. Map to Common Feature Schema

Defines the shared feature names and extracts them from both datasets. The common core includes all features that exist in both datasets at equivalent granularity (mean per observation, then mean across observations).

In [12]:
COMMON_FEATURES = [
    # S2 band means available in the default AOI cube export.
    "s2_B02_mean", "s2_B03_mean", "s2_B04_mean",
    "s2_B05_mean", "s2_B06_mean", "s2_B07_mean", "s2_B08_mean",
    "s2_B8A_mean", "s2_B11_mean",
    # Vegetation indices that can be derived from the default AOI bands.
    "s2_NDVI_mean", "s2_SAVI_mean", "s2_EVI_mean", "s2_GNDVI_mean",
    "s2_NDWI_GREEN_mean", "s2_NDRE_mean", "s2_NDMI_mean", "s2_MSI_mean",
    "s2_BSI_mean",
    # Weather
    # "weather_daily_Temp_mean_C", "weather_daily_Temp_max_C", "weather_daily_Temp_min_C",
    # "weather_daily_Total_prec_mm", "weather_daily_gdd_base10",
    # "weather_daily_rainy_day", "weather_daily_heavy_rain_day",
    # "weather_daily_heat_stress_day", "weather_daily_cold_stress_day",
    # "weather_7d_Total_prec_mm", "weather_7d_Temp_mean_C", "weather_7d_gdd_base10",
    # "weather_7d_rainy_days",
    # "weather_14d_Total_prec_mm", "weather_14d_Temp_mean_C", "weather_14d_gdd_base10",
    # "weather_14d_rainy_days",
    # "weather_30d_Total_prec_mm", "weather_30d_Temp_mean_C", "weather_30d_gdd_base10",
    # "weather_30d_rainy_days",
    # "weather_cum_Total_prec_mm", "weather_cum_gdd_base10", "weather_cum_rainy_days",
    # "weather_cum_heavy_rain_days", "weather_cum_heat_stress_days", "weather_cum_cold_stress_days",
]

# print(f"Common S2/VI/weather features: {len(COMMON_FEATURES)}")
print(f"Common S2/VI features: {len(COMMON_FEATURES)}")


Common S2/VI features: 18


In [13]:
# --- Extract from YieldSAT (obs_mean__{feat}) ---
yieldsat_feat_map = {}
for feat in COMMON_FEATURES:
    col = f"obs_mean__{feat}"
    if col in yieldsat_model.columns:
        yieldsat_feat_map[f"feat_{feat}"] = col

print(f"YieldSAT mapped features: {len(yieldsat_feat_map)}")

# --- Extract from Morocco (no suffix -- .mean() preserves column names) ---
morocco_feat_map = {}
for feat in COMMON_FEATURES:
    col = feat
    if col in morocco_model.columns:
        morocco_feat_map[f"feat_{feat}"] = col

print(f"Morocco mapped features: {len(morocco_feat_map)}")

# --- Find intersection ---
common_keys = sorted(set(yieldsat_feat_map.keys()) & set(morocco_feat_map.keys()))
print(f"Common mapped features: {len(common_keys)}")
print("Feature keys:", common_keys)


YieldSAT mapped features: 18
Morocco mapped features: 18
Common mapped features: 18
Feature keys: ['feat_s2_B02_mean', 'feat_s2_B03_mean', 'feat_s2_B04_mean', 'feat_s2_B05_mean', 'feat_s2_B06_mean', 'feat_s2_B07_mean', 'feat_s2_B08_mean', 'feat_s2_B11_mean', 'feat_s2_B8A_mean', 'feat_s2_BSI_mean', 'feat_s2_EVI_mean', 'feat_s2_GNDVI_mean', 'feat_s2_MSI_mean', 'feat_s2_NDMI_mean', 'feat_s2_NDRE_mean', 'feat_s2_NDVI_mean', 'feat_s2_NDWI_GREEN_mean', 'feat_s2_SAVI_mean']


## 7. Build Combined Dataset

Creates a combined model table with the common features, a consistent five-label schema, and a source indicator column.

In [14]:
yieldsat_combined = yieldsat_model[["label", "source_crop"] + [yieldsat_feat_map[k] for k in common_keys]].copy()
yieldsat_combined.columns = ["label", "source_crop"] + common_keys
yieldsat_combined["source"] = "yieldsat"

morocco_combined = morocco_model[["label", "source_crop"] + [morocco_feat_map[k] for k in common_keys]].copy()
morocco_combined.columns = ["label", "source_crop"] + common_keys
morocco_combined["source"] = "morocco"

combined = pd.concat([yieldsat_combined, morocco_combined], ignore_index=True)

INCLUDED_MODEL_LABELS = [
    "Wheat",
    "Corn",
    "Alfalfa",
    # "Other crop",
    # "Background",
]

excluded_rows = combined.loc[~combined["label"].isin(INCLUDED_MODEL_LABELS), ["source", "label", "source_crop"]]
excluded_summary = excluded_rows.value_counts().rename("rows").reset_index()
combined = combined.loc[combined["label"].isin(INCLUDED_MODEL_LABELS)].reset_index(drop=True)
combined = combined.replace([np.inf, -np.inf], np.nan).fillna(0)

BAND_FEATURES = [
    "feat_s2_B02_mean", "feat_s2_B03_mean", "feat_s2_B04_mean",
    "feat_s2_B05_mean", "feat_s2_B06_mean", "feat_s2_B07_mean",
    "feat_s2_B08_mean", "feat_s2_B8A_mean", "feat_s2_B11_mean",
]
VI_FEATURES = [
    "feat_s2_NDVI_mean", "feat_s2_SAVI_mean", "feat_s2_EVI_mean", "feat_s2_GNDVI_mean",
    "feat_s2_NDWI_GREEN_mean", "feat_s2_NDRE_mean", "feat_s2_NDMI_mean", "feat_s2_MSI_mean",
    "feat_s2_BSI_mean",
]
NORMALIZED_BAND_FEATURES = [f"{col}_brightness_norm" for col in BAND_FEATURES]
SELECTED_NORMALIZED_BAND_FEATURES = [
    "feat_s2_B06_mean_brightness_norm",
    "feat_s2_B07_mean_brightness_norm",
    "feat_s2_B08_mean_brightness_norm",
    "feat_s2_B8A_mean_brightness_norm",
]

brightness = combined[BAND_FEATURES].mean(axis=1).replace(0, np.nan)
for source_col, norm_col in zip(BAND_FEATURES, NORMALIZED_BAND_FEATURES, strict=True):
    combined[norm_col] = combined[source_col] / brightness
combined[NORMALIZED_BAND_FEATURES] = combined[NORMALIZED_BAND_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)

# Target-crop-only variant: uncomment Other crop/Background above to restore them.
MODEL_FEATURES = VI_FEATURES + SELECTED_NORMALIZED_BAND_FEATURES

print("Included model labels:", INCLUDED_MODEL_LABELS)
print("Excluded rows summary:")
print(excluded_summary.to_string(index=False) if len(excluded_summary) else "None")
print("Combined target-crop table:", combined.shape)
print("Label x source:")
print(pd.crosstab(combined["label"], combined["source"]))
print("Raw band features:", len(BAND_FEATURES))
print("Brightness-normalized band features available:", len(NORMALIZED_BAND_FEATURES))
print("Brightness-normalized band features used:", len(SELECTED_NORMALIZED_BAND_FEATURES))
print("Vegetation index features used:", len(VI_FEATURES))
print("Model features:", len(MODEL_FEATURES))


Included model labels: ['Wheat', 'Corn', 'Alfalfa']
Excluded rows summary:
  source      label source_crop  rows
 morocco Background   Bare soil   582
yieldsat Other crop     soybean   341
 morocco Other crop       Beets   239
 morocco Other crop    Potatoes   221
 morocco Background        Weed   219
 morocco Other crop        Peas   134
yieldsat Other crop    rapeseed   111
 morocco Other crop      Peanut    99
 morocco Other crop      Onions    98
 morocco Other crop    Tomatoes    74
 morocco Other crop   Sugarcane    60
 morocco Other crop  Fava Beans    30
 morocco Other crop     Parsley    20
 morocco Other crop        Rice    14
 morocco Background       Water    11
Combined target-crop table: (3942, 30)
Label x source:
source   morocco  yieldsat
label                     
Alfalfa      857         0
Corn         764       303
Wheat       1564       454
Raw band features: 9
Brightness-normalized band features available: 9
Brightness-normalized band features used: 4
Vegetation in

## 8. Train/Test Split

Stratified split preserving label proportions. The source indicator is kept for diagnostics only.

In [15]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(combined["label"])
feature_cols = MODEL_FEATURES
X = combined[feature_cols]

label_counts = combined["label"].value_counts()
sample_weights = (len(combined) / (len(label_counts) * combined["label"].map(label_counts))).astype(float)

X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, combined.index,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)

train_meta = combined.iloc[train_idx]
test_meta = combined.iloc[test_idx]
train_sample_weights = sample_weights.loc[train_idx].to_numpy()
full_sample_weights = sample_weights.to_numpy()

print("Feature count:", len(feature_cols))
print("Classes:", label_encoder.classes_.tolist())
print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Train label x source:")
print(pd.crosstab(train_meta["label"], train_meta["source"]))
print("Test label x source:")
print(pd.crosstab(test_meta["label"], test_meta["source"]))
print("Mean sample weight by label:")
print(sample_weights.groupby(combined["label"]).mean().sort_values(ascending=False))
print("Effective weighted class totals:")
print(sample_weights.groupby(combined["label"]).sum().sort_values(ascending=False))
print("Model feature columns:")
print(feature_cols)


Feature count: 13
Classes: ['Alfalfa', 'Corn', 'Wheat']
Train rows: 3153
Test rows: 789
Train label x source:
source   morocco  yieldsat
label                     
Alfalfa      686         0
Corn         606       247
Wheat       1236       378
Test label x source:
source   morocco  yieldsat
label                     
Alfalfa      171         0
Corn         158        56
Wheat        328        76
Mean sample weight by label:
label
Alfalfa    1.533256
Corn       1.231490
Wheat      0.651140
Name: label, dtype: float64
Effective weighted class totals:
label
Alfalfa    1314.0
Corn       1314.0
Wheat      1314.0
Name: label, dtype: float64
Model feature columns:
['feat_s2_NDVI_mean', 'feat_s2_SAVI_mean', 'feat_s2_EVI_mean', 'feat_s2_GNDVI_mean', 'feat_s2_NDWI_GREEN_mean', 'feat_s2_NDRE_mean', 'feat_s2_NDMI_mean', 'feat_s2_MSI_mean', 'feat_s2_BSI_mean', 'feat_s2_B06_mean_brightness_norm', 'feat_s2_B07_mean_brightness_norm', 'feat_s2_B08_mean_brightness_norm', 'feat_s2_B8A_mean_brightness_n

## 9. Train XGBoost

In [16]:
xgb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

print("Training XGBoost...")
xgb_model.fit(X_train, y_train, model__sample_weight=train_sample_weights)
xgb_pred = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred, average="macro")

print(f"XGBoost accuracy: {xgb_acc:.4f}")
print(f"XGBoost macro F1: {xgb_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, xgb_pred, labels=list(range(len(label_encoder.classes_))), target_names=label_encoder.classes_, digits=4, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, xgb_pred))



Training XGBoost...
XGBoost accuracy: 0.8834
XGBoost macro F1: 0.8827

Classification report:
              precision    recall  f1-score   support

     Alfalfa     0.8729    0.9240    0.8977       171
        Corn     0.8638    0.8598    0.8618       214
       Wheat     0.8987    0.8787    0.8886       404

    accuracy                         0.8834       789
   macro avg     0.8785    0.8875    0.8827       789
weighted avg     0.8837    0.8834    0.8833       789

Confusion matrix:
[[158   0  13]
 [  3 184  27]
 [ 20  29 355]]


## 10. Train LightGBM

In [17]:
lgb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median").set_output(transform="pandas")),
    ("model", LGBMClassifier(
        objective="multiclass",
        n_estimators=600,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1,
    )),
])

print("Training LightGBM...")
lgb_model.fit(X_train, y_train, model__sample_weight=train_sample_weights)
lgb_pred = lgb_model.predict(X_test)

lgb_acc = accuracy_score(y_test, lgb_pred)
lgb_f1 = f1_score(y_test, lgb_pred, average="macro")

print(f"LightGBM accuracy: {lgb_acc:.4f}")
print(f"LightGBM macro F1: {lgb_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, lgb_pred, labels=list(range(len(label_encoder.classes_))), target_names=label_encoder.classes_, digits=4, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, lgb_pred))



Training LightGBM...
LightGBM accuracy: 0.8821
LightGBM macro F1: 0.8792

Classification report:
              precision    recall  f1-score   support

     Alfalfa     0.9018    0.8596    0.8802       171
        Corn     0.8720    0.8598    0.8659       214
       Wheat     0.8795    0.9035    0.8913       404

    accuracy                         0.8821       789
   macro avg     0.8845    0.8743    0.8792       789
weighted avg     0.8823    0.8821    0.8820       789

Confusion matrix:
[[147   2  22]
 [  2 184  28]
 [ 14  25 365]]


## 11. Train CatBoost

In [18]:
cat_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", CatBoostClassifier(
        iterations=300,
        depth=5,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=RANDOM_SEED,
        verbose=False,
        allow_writing_files=False,
    )),
])

print("Training CatBoost...")
cat_model.fit(X_train, y_train, model__sample_weight=train_sample_weights)
cat_pred = cat_model.predict(X_test)

cat_acc = accuracy_score(y_test, cat_pred)
cat_f1 = f1_score(y_test, cat_pred, average="macro")

print(f"CatBoost accuracy: {cat_acc:.4f}")
print(f"CatBoost macro F1: {cat_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, cat_pred, labels=list(range(len(label_encoder.classes_))), target_names=label_encoder.classes_, digits=4, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, cat_pred))



Training CatBoost...
CatBoost accuracy: 0.8720
CatBoost macro F1: 0.8717

Classification report:
              precision    recall  f1-score   support

     Alfalfa     0.8457    0.9298    0.8858       171
        Corn     0.8378    0.8692    0.8532       214
       Wheat     0.9050    0.8490    0.8761       404

    accuracy                         0.8720       789
   macro avg     0.8629    0.8827    0.8717       789
weighted avg     0.8739    0.8720    0.8720       789

Confusion matrix:
[[159   0  12]
 [  4 186  24]
 [ 25  36 343]]


## 12. Model Comparison

In [19]:
comparison = pd.DataFrame([
    {"model": "XGBoost", "accuracy": xgb_acc, "macro_f1": xgb_f1},
    {"model": "LightGBM", "accuracy": lgb_acc, "macro_f1": lgb_f1},
    {"model": "CatBoost", "accuracy": cat_acc, "macro_f1": cat_f1},
]).sort_values("macro_f1", ascending=False).reset_index(drop=True)

display(comparison)
print(f"Best model by macro F1: {comparison.iloc[0]['model']}")

,model,accuracy,macro_f1
0,XGBoost,0.883397,0.882722
1,LightGBM,0.882129,0.879151
2,CatBoost,0.871990,0.871707


Best model by macro F1: XGBoost


## 13. Per-Class Metrics for Best Model

In [20]:
best_name = comparison.iloc[0]["model"]
pred_map = {"XGBoost": xgb_pred, "LightGBM": lgb_pred, "CatBoost": cat_pred}
best_pred = pred_map[best_name]

print(f"Per-class metrics for {best_name}:")
report = classification_report(y_test, best_pred, labels=list(range(len(label_encoder.classes_))), target_names=label_encoder.classes_, digits=4, output_dict=True)
per_class = pd.DataFrame(report).transpose()
display(per_class)


Per-class metrics for XGBoost:


,precision,recall,f1-score,support
Alfalfa,0.872928,0.923977,0.897727,171.000000
Corn,0.863850,0.859813,0.861827,214.000000
Wheat,0.898734,0.878713,0.888611,404.000000
accuracy,0.883397,0.883397,0.883397,0.883397
macro avg,0.878504,0.887501,0.882722,789.000000
weighted avg,0.883680,0.883397,0.883322,789.000000


## 14. Feature Importance (Best Model)

Shows the top features from the best model as a quick diagnostic.

In [21]:
best_pipeline = {"XGBoost": xgb_model, "LightGBM": lgb_model, "CatBoost": cat_model}[best_name]
best_inner = best_pipeline.named_steps["model"]

if hasattr(best_inner, "feature_importances_"):
    importances = best_inner.feature_importances_
    top_n = 20
    top_idx = np.argsort(importances)[::-1][:top_n]
    print(f"Top {top_n} features ({best_name}):")
    for i in top_idx:
        print(f"  {feature_cols[i]:50s} {importances[i]:.6f}")

Top 20 features (XGBoost):
  feat_s2_SAVI_mean                                  0.155758
  feat_s2_NDVI_mean                                  0.104604
  feat_s2_BSI_mean                                   0.097411
  feat_s2_EVI_mean                                   0.077998
  feat_s2_B07_mean_brightness_norm                   0.076062
  feat_s2_NDMI_mean                                  0.072723
  feat_s2_NDRE_mean                                  0.071998
  feat_s2_NDWI_GREEN_mean                            0.070855
  feat_s2_B06_mean_brightness_norm                   0.068734
  feat_s2_GNDVI_mean                                 0.067325
  feat_s2_MSI_mean                                   0.060114
  feat_s2_B8A_mean_brightness_norm                   0.052708
  feat_s2_B08_mean_brightness_norm                   0.023709


## 15. Cross-Source Evaluation

Evaluates the best combined model separately on YieldSAT-only and Morocco-only test samples to check for source bias.

In [22]:
for source_name in ["yieldsat", "morocco"]:
    mask = test_meta["source"] == source_name
    if mask.sum() == 0:
        print(f"No {source_name} samples in test set")
        continue
    src_pred = best_pred[mask.values]
    src_true = y_test[mask.values]
    src_acc = accuracy_score(src_true, src_pred)
    src_f1 = f1_score(src_true, src_pred, average="macro")
    print(f"\n{source_name} test subset ({mask.sum()} rows):")
    print(f"  Accuracy: {src_acc:.4f}, Macro F1: {src_f1:.4f}")
    src_labels = label_encoder.inverse_transform(src_true)
    src_pred_labels = label_encoder.inverse_transform(src_pred)
    print(classification_report(src_true, src_pred, labels=list(range(len(label_encoder.classes_))), target_names=label_encoder.classes_, digits=4, zero_division=0))



yieldsat test subset (132 rows):
  Accuracy: 0.9167, Macro F1: 0.9159
              precision    recall  f1-score   support

     Alfalfa     0.0000    0.0000    0.0000         0
        Corn     0.8571    0.9643    0.9076        56
       Wheat     0.9710    0.8816    0.9241        76

    accuracy                         0.9167       132
   macro avg     0.6094    0.6153    0.6106       132
weighted avg     0.9227    0.9167    0.9171       132


morocco test subset (657 rows):
  Accuracy: 0.8767, Macro F1: 0.8742
              precision    recall  f1-score   support

     Alfalfa     0.8729    0.9240    0.8977       171
        Corn     0.8667    0.8228    0.8442       158
       Wheat     0.8834    0.8780    0.8807       328

    accuracy                         0.8767       657
   macro avg     0.8743    0.8749    0.8742       657
weighted avg     0.8767    0.8767    0.8764       657



## 16. Train Best Model on Full Data and Save

Retrains the best-performing model on the entire combined dataset (no train/test split) and saves the artifact for later inference.

In [24]:
import joblib
from pathlib import Path

best_name = comparison.iloc[0]["model"]
print(f"Retraining {best_name} on full combined dataset ({len(X)} rows)...")

if best_name == "XGBoost":
    full_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=RANDOM_SEED, n_jobs=-1,
        )),
    ])
elif best_name == "LightGBM":
    full_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median").set_output(transform="pandas")),
        ("model", LGBMClassifier(
            objective="multiclass", n_estimators=600, learning_rate=0.03,
            num_leaves=31, max_depth=-1, subsample=0.9, colsample_bytree=0.9,
            random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1,
        )),
    ])
elif best_name == "CatBoost":
    full_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", CatBoostClassifier(
            iterations=300, depth=5, learning_rate=0.05,
            loss_function="MultiClass", random_seed=RANDOM_SEED,
            verbose=False, allow_writing_files=False,
        )),
    ])

full_pipeline.fit(X, y, model__sample_weight=full_sample_weights)

artifact_dir = Path("data/models/combined_crop_classifier")
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "model.pkl"
joblib.dump(full_pipeline, model_path)
print(f"Model saved: {model_path}")

encoder_path = artifact_dir / "label_encoder.pkl"
joblib.dump(label_encoder, encoder_path)
print(f"Label encoder saved: {encoder_path}")

features_path = artifact_dir / "feature_columns.pkl"
joblib.dump(feature_cols, features_path)
print(f"Feature columns saved: {features_path}")

print(f"\nArtifact summary:")
print(f"  Best model: {best_name}")
print(f"  Classes: {label_encoder.classes_.tolist()}")
print(f"  Features: {len(feature_cols)}")
# print(f"  Other crop weight multiplier: {OTHER_CROP_WEIGHT_MULTIPLIER}")



Retraining XGBoost on full combined dataset (3942 rows)...
Model saved: data\models\combined_crop_classifier\model.pkl
Label encoder saved: data\models\combined_crop_classifier\label_encoder.pkl
Feature columns saved: data\models\combined_crop_classifier\feature_columns.pkl

Artifact summary:
  Best model: XGBoost
  Classes: ['Alfalfa', 'Corn', 'Wheat']
  Features: 13
